# **Xây dựng mô hình dự đoán tỷ lệ tử vong của ICU từ MIMIC-IV**

**Mục tiêu:**

Notebook này thực hiện xây dựng một mô hình dự đoán tỷ lệ tử vong từ bảng MIMIC bằng các mô hình học máy. Nhằm tạo ra một chuẩn dữ liệu để so sánh với các kỹ thuật khác trong tương lai (nếu có)

**Bộ dữ liệu**

MIMIC-IV là cơ sở dữ liệu lớn gồm các bảng về thông tin bệnh nhân, lâm sàng, các thủ thuật, v...

Lý do chọn bộ dữ liệu này do tính phức tạp, sát với dữ liệu thực tế, việc xử lý dữ liệu này là một trong những bước khó khắn, ...

- 2 module chính được sử dụng trong notebook này chính là `hosp` và `icu`. Chi tiết các bảng lựa chọn sẽ được mô tả ở dưới


## **1. Nạp thư viện và dữ liệu cần thiết**

In [49]:
# Bỏ comment nếu chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')
!pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

from typing import List, Tuple, Dict, Optional

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Libraries loaded


In [50]:
DATA_DIR = "/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1"  # Google Colab

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()


## **2. Tiền xử lý dữ liệu trước khi chia huấn luyện**

### **2.1. Xây dựng tập cohort**

Cohort được xây dựng bám sát theo các tiêu chí như sau:
- Phải là người trưởng thành (anchor_age > 18)
- Thời gian nằm ICU phải trên 24h (LOS >= 24)
- Chỉ lấy lần nhập ICU đầu tiên trong một lần nhập viện
- Loại các bệnh nhân đã tử vong, xuât hoặc chuyển viện trong vòng 24h đầu

In [21]:
def build_and_val_icu_cohort(patients_df: pd.DataFrame, admissions_df: pd.DataFrame, icustays_df: pd.DataFrame) -> pd.DataFrame:
  # Chuyển đổi định dạng mốc thời gian sang Datetime
  time_cols = {
      'icustays_df': ['intime', 'outtime'],
      'admissions_df': ['admittime', 'dischtime', 'edregtime', 'edouttime'],
      'patients_df' : []
  }

  for col in time_cols['icustays_df']:
        icustays_df[col] = pd.to_datetime(icustays_df[col])
  for col in time_cols['admissions_df']:
        admissions_df[col] = pd.to_datetime(admissions_df[col])
  # 1. Inner Join với patients
  cohort = icustays_df.merge(
  admissions_df[["subject_id", "hadm_id", "admittime", "dischtime", "admission_type",
                        "admission_location", "deathtime",
                        "hospital_expire_flag"]],
          on=['subject_id', 'hadm_id'],
          how='inner'
      )
  # 2. Inner join với admission
  cohort = cohort.merge(
          patients_df[["subject_id", "gender", "anchor_age"]],
          on='subject_id',
          how='inner'
      )

  initial_count = len(cohort)
  print(f"Tổng số lượt ICU ban đầu: {initial_count:,}")

  # Áp dụng các tiêu chí lọc
  # ---------------------------------------------
  # Tiêu chí 1: Người trưởng thành (>= 18 tuổi)
  # ----------------------------------------------
  cohort = cohort[cohort["anchor_age"] >= 18]
  print(f"-> Sau khi lọc người trưởng thành (>=18t): {len(cohort):,} ca")

  # ---------------------------------------------
  # Tiêu chí 2: Lấy ca ICU đầu tiên của mỗi bệnh nhân (First ICU stay per patient)
  # Sắp xếp theo intime để chắc chắn lấy ca đầu tiên trong đời/lịch sử của bệnh nhân
  # ----------------------------------------------
  cohort = (
      cohort.sort_values(["subject_id", "intime"])
            .groupby("subject_id", as_index=False)
            .first()
  )

  print(f"-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: {len(cohort):,} ca")

  # Tính thời gian nằm ICU (tính theo ngày)
  cohort["icu_los_days"] = (
      pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
  ).dt.total_seconds() / 86400
  cohort["icu_los_hours"] = cohort["icu_los_days"] * 24

  # ---------------------------------------------
  # Tiêu chí 3: ICU stay trên 1 ngày (>= 1 ngày, tức >= 24 giờ)
  # ---------------------------------------------
  cohort = cohort[cohort["icu_los_days"] >= 1.0]
  print(f"-> Sau khi lọc ICU stay > 1 ngày: {len(cohort):,} ca")

  # Tạo mốc thời gian kết thúc cửa sổ quan sát (intime + 24h)
  cohort['obs_end_time'] = cohort['intime'] + pd.Timedelta(hours=24)

  # Xây dựng nhóm tuổi
  cohort["age_group"] = pd.cut(
      cohort["anchor_age"],
      bins=[17, 30, 50, 65, 80, 120],
      labels=["18-30", "31-50", "51-65", "66-80", "80+"]
  )

  print("=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===")
  """
  Tiêu chí loại bỏ các lượt ICU vi phạm logic:
  - Tiêu chí 1: Các khóa phải là duy nhất
  - Tiêu chí 2: Kiểm tra thứ tự các môc thời gian logic
    - Thời gian nhập viện <= Thời gian vào ICU < Thời gian vào ICU + 24h <= Thời gian ra ICU <= Thời gian ra viện
  - Tiêu chí 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
    - Thời điểm tử vong của bệnh nhân phải lớn hơn thười gian vào ICU + 24h

  """

  # List lưu giữ các stay_id vi phạm cần loại bỏ
  invalid_stay_ids = set()

  # Check 1: Kiểm tra tính duy nhất của Khóa
  total_rows = len(cohort)
  unique_stays = cohort['stay_id'].nunique()
  print(f"Tổng số dòng: {total_rows} | Số stay_id duy nhất: {unique_stays}")
  if total_rows != unique_stays:
      print("Bị trùng lặp stay_id do phép Join! Cần loại bỏ bản ghi trùng.")
      cohort = cohort.drop_duplicates(subset=['stay_id'])

  # Check 2: Kiểm tra thứ tự mốc thời gian logic
  invalid_time_mask = (
        (cohort['admittime'] > cohort['intime']) |
        (cohort['intime'] >= cohort['outtime']) |
        (cohort['outtime'] > cohort['dischtime'])
    )
  time_faulty_ids = cohort[invalid_time_mask]['stay_id'].tolist()
  invalid_stay_ids.update(time_faulty_ids)
  print(f"Phát hiện {len(time_faulty_ids)} lượt ICU vi phạm thứ tự thời gian sinh lý.")

  # Check 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
  early_death_mask = (
        (cohort['hospital_expire_flag'] == 1) &
        (cohort['deathtime'].notna()) &
        (cohort['deathtime'] <= cohort['obs_end_time'])
    )
  early_death_ids = cohort[early_death_mask]['stay_id'].tolist()
  invalid_stay_ids.update(early_death_ids)
  print(f"Phát hiện {len(early_death_ids)} bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.")

  # Loai bỏ các bản ghi vi phạm khỏi Cohort chính thức
  clean_cohort = cohort[~cohort['stay_id'].isin(invalid_stay_ids)].reset_index(drop=True)

  print(f"=== KHỞI TẠO COHORT THÀNH CÔNG ===")
  print(f"Số lượng bệnh nhân hợp lệ cuối cùng: {len(clean_cohort)}")
  print(f"Thời gian nằm ICU trung bình: {cohort['icu_los_days'].mean():.2f} ngày")
  print(f"Tỷ lệ tử vong (Mortality Rate): {clean_cohort['hospital_expire_flag'].mean():.2%}")

  return clean_cohort


In [22]:
print("Loading core tables...")
patients   = load("hosp", "patients.csv.gz")      # Thông tin bệnh nhân
admissions = load("hosp", "admissions.csv.gz")    # Thông tin nhập viện
icustays   = load("icu",  "icustays.csv.gz")      # Thông tin lần nằm

cohort = build_and_val_icu_cohort(patients, admissions, icustays)

Loading core tables...
  ✓ hosp/patients.csv.gz: 364,627 rows × 6 cols
  ✓ hosp/admissions.csv.gz: 546,028 rows × 16 cols
  ✓ icu/icustays.csv.gz: 94,458 rows × 8 cols
Tổng số lượt ICU ban đầu: 94,458
-> Sau khi lọc người trưởng thành (>=18t): 94,458 ca
-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: 65,366 ca
-> Sau khi lọc ICU stay > 1 ngày: 51,839 ca
=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===
Tổng số dòng: 51839 | Số stay_id duy nhất: 51839
Phát hiện 8714 lượt ICU vi phạm thứ tự thời gian sinh lý.
Phát hiện 171 bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.
=== KHỞI TẠO COHORT THÀNH CÔNG ===
Số lượng bệnh nhân hợp lệ cuối cùng: 43124
Thời gian nằm ICU trung bình: 4.26 ngày
Tỷ lệ tử vong (Mortality Rate): 4.39%


In [23]:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,admission_type,admission_location,deathtime,hospital_expire_flag,gender,anchor_age,icu_los_days,icu_los_hours,obs_end_time,age_group
0,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,2150-11-02 18:02:00,2150-11-12 13:45:00,EW EMER.,EMERGENCY ROOM,None,0,F,86,3.893252,93.438056,2150-11-03 19:37:00,80+
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,EW EMER.,EMERGENCY ROOM,None,0,F,55,1.118032,26.832778,2157-11-21 19:18:02,51-65
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,2110-04-11 15:08:00,2110-04-14 15:00:00,EW EMER.,PACU,None,0,F,46,1.338588,32.126111,2110-04-12 15:52:22,31-50
3,10002013,23581541,39060235,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2160-05-18 10:00:53,2160-05-19 17:33:33,1.314352,2160-05-18 07:45:00,2160-05-23 13:30:00,SURGICAL SAME DAY ADMISSION,PHYSICIAN REFERRAL,None,0,F,53,1.314352,31.544444,2160-05-19 10:00:53,51-65
4,10002114,27793700,34672098,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2162-02-17 23:30:00,2162-02-20 21:16:27,2.907257,2162-02-17 22:32:00,2162-03-04 15:16:00,OBSERVATION ADMIT,PHYSICIAN REFERRAL,None,0,M,56,2.907257,69.774167,2162-02-18 23:30:00,51-65


In [24]:
cohort.isna().sum()

,0
subject_id,0
hadm_id,0
stay_id,0
first_careunit,0
last_careunit,0
intime,0
outtime,0
los,0
admittime,0
dischtime,0


**Có missing cần lưu ý**
- `deathtime`: Có thể `Null - None` do bệnh nhân không tử vong

In [39]:
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'admission_type',
       'admission_location', 'deathtime', 'hospital_expire_flag', 'gender',
       'anchor_age', 'icu_los_days', 'icu_los_hours', 'obs_end_time',
       'age_group'],
      dtype='object')

### **2.2. Chia tập train - test - validate để huấn luyện mô hình & Xác định nhãn cho bài toán**

- Thực hiện chia tập dữ liệu với tỷ lệ tử vong ở mỗi tập là như nhau
  - Chia theo tý lệ train (70%) - test (20%) - val (10%)
- **Nhãn mục tiêu** `hospital_expire_flag`
  - Với `0` là bệnh nhân sống sót và `1` là bệnh nhân tử vong

In [41]:
# Sinh x và y
X = cohort.drop(columns=['hospital_expire_flag'])
y = cohort['hospital_expire_flag']

In [45]:
from sklearn.model_selection import train_test_split

def split_data(
    X: pd.DataFrame,
    y: pd.Series,
    target_col: str = 'hospital_expire_flag',
    test_size: float = 0.2,
    val_size: float = 0.125,
    random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Chia dataset thành 3 tập Train/Val/Test theo `subject_id` để chống Data Leakage.
    Tỷ lệ mặc định: 70% Train - 20% Val - 10% Test.
    """
    # Tách 20% cho tập Test (Còn lại 80% cho Train + Val)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # Chia 80% đó thành Train (70% tổng) và Val (10% tổng)
    # Tỷ lệ tập Val trong tập Train_Val là: 10% / 80% = 0.125 (12.5%)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=val_size,
        stratify=y_train_val,
        random_state=random_state
    )

    # 4. Kiểm tra kích thước các tập
    print(f"Kích thước tập Train: {X_train.shape[0]} ({len(X_train)/len(cohort):.0%})")
    print(f"Kích thước tập Val:   {X_val.shape[0]} ({len(X_val)/len(cohort):.0%})")
    print(f"Kích thước tập Test:  {X_test.shape[0]} ({len(X_test)/len(cohort):.0%})")

    # 5. Kiểm tra tỷ lệ biến mục tiêu có được giữ nguyên không
    print("\nTỷ lệ nhãn trong tập Train:\n", y_train.value_counts(normalize=True))
    print("\nTỷ lệ nhãn trong tập Val:\n", y_val.value_counts(normalize=True))
    print("\nTỷ lệ nhãn trong tập Test:\n", y_test.value_counts(normalize=True))

    return X_train_val, X_test, y_train_val, y_test, X_train, X_val, y_train, y_val

## **3. Xây dựng các bảng đặc trưng thực hiện tiền xử lý cho dữ liệu huấn luyện**

In [44]:
from typing import Dict, Tuple, List

### **3.1. Xây dựng bảng chart vitals từ `chartevent`**

Kết hợp với bảng cohort với cửa số được lấy trong vòng 24h khi nằm ở icu
- intime <= charttime <= intime + 24h
- Lọc các chỉ số theo mã id hợp lệ rồi học các giá trị ngoại lai
- Thực hiện sinh thêm các đặc trưng mới với các chỉ số phức tạp

In [42]:
def extract_clean_chartevents(
    events_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    item_mapping: Dict[int, str],
    outlier_bounds: Dict[str, Tuple[float, float]]
) -> pd.DataFrame:
    """
    Lọc dữ liệu chartevents theo stay_id trong 24h ICU và làm sạch nhiễu sinh lý.
    """
    # Lọc trước các itemid hợp lệ để tối ưu bộ nhớ
    df = events_df[events_df['itemid'].isin(item_mapping.keys())].copy()
    df['feature_name'] = df['itemid'].map(item_mapping)

    # Merge lấy intime từ cohort để khóa cửa sổ 24h
    df = df.merge(
        cohort_df[['stay_id', 'intime']],
        on='stay_id',
        how='inner'
    )

    # Lọc thời gian: intime <= charttime <= intime + 24h
    df['charttime'] = pd.to_datetime(df['charttime'])
    df['intime'] = pd.to_datetime(df['intime'])
    df['delta_hours'] = (df['charttime'] - df['intime']).dt.total_seconds() / 3600.0

    mask_time = (df['delta_hours'] >= 0) & (df['delta_hours'] <= 24.0)
    df = df[mask_time].copy()

    # Chuyển đổi dữ liệu đo về dạng số
    df['valuenum'] = pd.to_numeric(df['valuenum'], errors='coerce')
    df = df.dropna(subset=['valuenum']).copy()

    # Quy đổi đơn vị đo chuẩn (F -> C, lbs -> kg, inches -> cm)
    if 'valueuom' in df.columns:
        uom_str = df['valueuom'].astype(str).str.lower()

        # Độ F -> C
        f_mask = uom_str.str.contains('f', na=False) | (df['feature_name'].str.contains('temp', case=False) & (df['valuenum'] > 50))
        df.loc[f_mask, 'valuenum'] = (df.loc[f_mask, 'valuenum'] - 32.0) * 5.0 / 9.0

        # lbs -> kg
        lbs_mask = uom_str.str.contains('lb', na=False)
        df.loc[lbs_mask, 'valuenum'] = df.loc[lbs_mask, 'valuenum'] * 0.453592

        # inches -> cm
        inch_mask = uom_str.str.contains('inch|in', na=False)
        df.loc[inch_mask, 'valuenum'] = df.loc[inch_mask, 'valuenum'] * 2.54

    # Lọc nhiễu sinh lý (Outliers)
    for feat_name, (min_val, max_val) in outlier_bounds.items():
        feat_mask = df['feature_name'] == feat_name
        invalid_mask = feat_mask & ((df['valuenum'] < min_val) | (df['valuenum'] > max_val))
        df.loc[invalid_mask, 'valuenum'] = np.nan

    df = df.dropna(subset=['valuenum']).copy()

    # Sắp xếp lại dữ liệu theo ID và mốc thời gian
    cols_to_keep = ['stay_id', 'charttime', 'delta_hours', 'feature_name', 'valuenum']
    df = df.sort_values(by=['stay_id', 'feature_name', 'charttime'])[cols_to_keep]

    return df.reset_index(drop=True)

# **Baseline 2:**

In [54]:
def build_cohort(data_dir):
    patients = pd.read_csv(
        os.path.join(data_dir, "hosp", "patients.csv.gz"),
        usecols=["subject_id", "gender", "anchor_age", "dod"],
        parse_dates=["dod"],
    )
    admissions = pd.read_csv(
        os.path.join(data_dir, "hosp", "admissions.csv.gz"),
        usecols=["subject_id", "hadm_id", "admittime", "dischtime",
                 "deathtime", "hospital_expire_flag"],
        parse_dates=["admittime", "dischtime", "deathtime"],
    )
    icustays = pd.read_csv(
        os.path.join(data_dir, "icu", "icustays.csv.gz"),
        usecols=["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los"],
        parse_dates=["intime", "outtime"],
    )

    icustays = (
        icustays.sort_values(["subject_id", "hadm_id", "intime"])
        .groupby(["subject_id", "hadm_id"], as_index=False)
        .first()
    )
    cohort = (
        icustays
        .merge(admissions, on=["subject_id", "hadm_id"], how="inner")
        .merge(patients,   on="subject_id",               how="inner")
    )
    cohort = cohort[cohort["anchor_age"] >= 18].copy()
    cohort["Outcome"]   = cohort["hospital_expire_flag"].astype(int)
    cohort["Sex"]       = (cohort["gender"] == "M").astype(int)
    cohort["LOS_hours"] = cohort["los"].astype(float) * 24
    cohort = cohort.rename(columns={
        "subject_id": "PatientID", "hadm_id": "AdmissionID",
        "stay_id": "StayID",       "intime":  "ICUInTime",
        "outtime": "ICUOutTime",   "anchor_age": "Age",
    })[["PatientID", "AdmissionID", "StayID",
        "ICUInTime", "ICUOutTime",
        "Age", "Sex", "Outcome", "LOS_hours"]].reset_index(drop=True)

    print(f"  ICU stays      : {len(cohort):,}")
    print(f"  Unique patients: {cohort['PatientID'].nunique():,}")
    print(f"  Mortality      : {cohort['Outcome'].sum():,}  "
             f"({cohort['Outcome'].mean()*100:.1f}%)")
    return cohort

In [55]:
cohort = build_cohort(DATA_DIR)

  ICU stays      : 85,242
  Unique patients: 65,366
  Mortality      : 9,475  (11.1%)
